In [ ]:
!pip install sentence-transformers chromadb openai

In [ ]:
# Our knowledge base — 5 facts about AI
documents = [
    "AI can learn patterns from data.",
    "Machine learning is a subset of AI.",
    "Deep learning uses neural networks.",
    "LLMs can understand and generate text.",
    "RAG combines retrieval with language generation."
]

print("Number of documents:", len(documents))

Number of documents: 5


In [ ]:
# For now each document is already a chunk
# In real RAG these would be paragraphs split from larger docs
chunks = documents  # this should be the documents list itself

print("Number of chunks:", len(chunks))  # use len()
print("Example chunk:", chunks[0])       # use chunks[0]

Number of chunks: 5
Example chunk: AI can learn patterns from data.


In [ ]:
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed all chunks
chunk_embeddings = model.encode(chunks)

print("Embeddings shape:", chunk_embeddings.shape)
print("Each chunk is now a vector of", chunk_embeddings.shape[1], "numbers")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape: (5, 384)
Each chunk is now a vector of 384 numbers


In [ ]:
import chromadb

# Create client
client = chromadb.Client()

# Create collection
collection = client.create_collection(
    name="ai_knowledge",
    metadata={"hnsw:space": "cosine"}
)

# Add chunks + embeddings
collection.add(
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    ids=[f"doc_{i}" for i in range(len(chunks))]
)

print(f"Stored {collection.count()} chunks in ChromaDB")

Stored 5 chunks in ChromaDB


In [ ]:
# User's question
question = "What is deep learning?"

# Embed the question
question_embedding = model.encode(question).tolist()

# Query ChromaDB for top 2 most similar chunks
results = collection.query(
    query_embeddings=[question_embedding],
    n_results=2
)

print("Question:", question)
print("\nRetrieved chunks:")
for i, doc in enumerate(results['documents'][0]):
    score = 1 - results['distances'][0][i]
    print(f"{i+1}. '{doc}' (similarity: {score:.4f})")

Question: What is deep learning?

Retrieved chunks:
1. 'Deep learning uses neural networks.' (similarity: 0.7275)
2. 'Machine learning is a subset of AI.' (similarity: 0.5110)


In [ ]:
import os
os.environ["GROQ_API_KEY"] = "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"
print("API key set!")

API key set!


In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.2 MB/s eta 0:00:00


In [ ]:
from groq import Groq

# Initialize Groq client
client_llm = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Build context from retrieved chunks
context = "\n".join(results['documents'][0])

# Build RAG prompt
prompt = f"""Answer the question based only on the context below.
Context:
{context}

Question:
{question}

Answer:"""

# Send to LLM
response = client_llm.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

# Print answer
print("Question:", question)
print("\nContext used:")
for doc in results['documents'][0]:
    print(f"  - {doc}")
print("\nGenerated Answer:", response.choices[0].message.content)

Question: What is deep learning?

Context used:
  - Deep learning uses neural networks.
  - Machine learning is a subset of AI.

Generated Answer: Deep learning is a branch of machine learning that uses neural networks.
